# CineMatch — Letterboxd Comfort / Feel-Good Scraper

**Owner:** Geoff (CineMatch — DMW track)  
**Goal:** Scrape 200+ unique films across community-curated *Comfort* and *Feel-Good* Letterboxd lists, plus full per-film metadata, into a single CSV.  
**Output:** `data/comfort_feelgood_films.csv`

---
## Run order
1. **§1 Install** (one time)
2. **§2 Imports & config**
3. **§3 HTTP helper**
4. **§4 DIAGNOSTIC** ← run this first if anything is broken
5. **§5–§8** — list scraping + film scraping
6. **§9 Full run**
7. **§10 Inspect**


## §1. Install dependencies

In [1]:
%pip install --quiet requests beautifulsoup4 pandas lxml


[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## §2. Imports & config
**Edit `LISTS_TO_SCRAPE` here if you want to swap lists.** That's the only thing you should need to change.


In [2]:
import json
import re
import time
import csv
from datetime import datetime, timezone
from pathlib import Path

import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE_URL = "https://letterboxd.com"

# (label, url) -- label appears in the CSV's `lists` column. Keep trailing slash.
LISTS_TO_SCRAPE = [
    ("ultimate_comfort", "https://letterboxd.com/cinemaxwell/list/the-ultimate-comfort-movie-list/"),
    ("comfort_izzy",     "https://letterboxd.com/cinemaparadisa/list/comfort-movies/"),
    ("empire_feelgood",  "https://letterboxd.com/johnlees/list/empires-feelgood-films-list/"),
    ("heartwarming",     "https://letterboxd.com/rosh_rko/list/heartwarming-feel-good-films-essential-viewing/"),
    ("comfort_boze",     "https://letterboxd.com/sketchesbyboze/list/comfort-films/"),
]

DELAY_SECONDS = 1.5  # don't lower -- Letterboxd will rate-limit you (HTTP 429)

# Beefed-up headers that look like a real Chrome browser.
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Cache-Control": "no-cache",
    "Pragma": "no-cache",
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "none",
    "Sec-Fetch-User": "?1",
    "Upgrade-Insecure-Requests": "1",
}

OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / "comfort_feelgood_films.csv"
DEBUG_DIR = Path("debug")
DEBUG_DIR.mkdir(exist_ok=True)

session = requests.Session()
session.headers.update(HEADERS)

print("Config OK.")
print(f"Lists to scrape: {len(LISTS_TO_SCRAPE)}")
print(f"Output file:     {OUTPUT_CSV}")
print(f"Debug dumps:     {DEBUG_DIR}/")


Config OK.
Lists to scrape: 5
Output file:     data/comfort_feelgood_films.csv
Debug dumps:     debug/


## §3. HTTP helper
Never use `requests.get` directly — always go through `fetch()` so you get the politeness delay.

In [3]:
def fetch(url, retries=3):
    """GET with politeness delay + retries. Returns BeautifulSoup or None."""
    for attempt in range(retries):
        try:
            r = session.get(url, timeout=30)
            if r.status_code == 200:
                time.sleep(DELAY_SECONDS)
                return BeautifulSoup(r.text, "html.parser")
            if r.status_code == 404:
                print(f"  [404] {url}")
                return None
            print(f"  [{r.status_code}] {url}  (attempt {attempt + 1}/{retries})")
        except requests.RequestException as exc:
            print(f"  [err] {url} -- {exc}")
        time.sleep(DELAY_SECONDS * (attempt + 2))
    return None


def fetch_raw(url):
    """Like fetch() but returns the full Response object (for diagnostics)."""
    return session.get(url, timeout=30)


## §4. DIAGNOSTIC — run this if anything is breaking

This cell does **everything** you need to figure out why scraping isn't working:
- Hits a list page
- Prints HTTP status, content type, page size
- Saves the raw HTML to `debug/list_page.html` so you can open it in a browser
- Prints the first 1500 chars of HTML so you can see if it's a real page or a bot challenge
- Counts how many elements match every plausible Letterboxd selector
- Lists every distinct `data-*` attribute it finds

**If posters_count = 0 below, paste the WHOLE output back to me and I can fix it.**


In [4]:
# DIAGNOSTIC — gather everything we need to debug
test_label, test_url = LISTS_TO_SCRAPE[0]
diag_url = test_url.rstrip("/") + "/page/1/"
print(f"Diagnostic target: {diag_url}\n")

# 1. Raw HTTP-level info
r = fetch_raw(diag_url)
print(f"HTTP status:     {r.status_code}")
print(f"Final URL:       {r.url}")
print(f"Content-Type:    {r.headers.get('Content-Type')}")
print(f"Content-Length:  {r.headers.get('Content-Length', 'not set')}")
print(f"Body bytes:      {len(r.content)}")
print(f"Body chars:      {len(r.text)}")
print(f"Server:          {r.headers.get('Server')}")

# 2. Save raw HTML for inspection
debug_path = DEBUG_DIR / "list_page.html"
debug_path.write_text(r.text, encoding="utf-8")
print(f"\nSaved raw HTML to: {debug_path.resolve()}")
print("Open that file in a browser to see exactly what came back.\n")

# 3. Show first 1500 chars
print("=" * 60)
print("FIRST 1500 CHARS OF HTML:")
print("=" * 60)
print(r.text[:1500])
print("=" * 60)

# 4. Parse and try every selector we know
soup = BeautifulSoup(r.text, "html.parser")

print("\nSelector match counts:")
selectors_to_try = [
    "div.film-poster[data-film-slug]",
    "div.film-poster",
    "li.poster-container",
    "li.poster-container div[data-film-slug]",
    "[data-film-slug]",
    "[data-target-link]",
    "[data-film-id]",
    "div[data-component-class='globals.comps.PosterComponent']",
    "div[data-component-class*='Poster']",
    "a[href^='/film/']",
    "img.image",
    "li.posteritem",
    "div.poster",
    "react-component",
]
for sel in selectors_to_try:
    try:
        n = len(soup.select(sel))
    except Exception as e:
        n = f"ERROR: {e}"
    print(f"  {sel!r:60s} -> {n}")

# 5. Find every distinct data-* attribute used on the page
print("\nAll distinct data-* attributes seen on the page:")
data_attrs = {}
for tag in soup.find_all(True):
    for attr_name in tag.attrs:
        if attr_name.startswith("data-"):
            data_attrs.setdefault(attr_name, 0)
            data_attrs[attr_name] += 1
for name, count in sorted(data_attrs.items(), key=lambda x: -x[1])[:30]:
    print(f"  {name:40s} (count: {count})")

# 6. Title of the page
title_tag = soup.select_one("title")
print(f"\n<title>: {title_tag.get_text(strip=True) if title_tag else '<none>'}")

# 7. Count anchors that look like film links
film_anchors = soup.select("a[href^='/film/']")
print(f"\nAnchors with href starting '/film/': {len(film_anchors)}")
if film_anchors:
    print("First 5 such anchors:")
    for a in film_anchors[:5]:
        print(f"  href={a.get('href')!r}")


Diagnostic target: https://letterboxd.com/cinemaxwell/list/the-ultimate-comfort-movie-list/page/1/

HTTP status:     200
Final URL:       https://letterboxd.com/cinemaxwell/list/the-ultimate-comfort-movie-list/page/1/
Content-Type:    text/html;charset=utf-8
Content-Length:  not set
Body bytes:      250385
Body chars:      250343
Server:          cloudflare

Saved raw HTML to: /Users/adonisgeoffmacias/Desktop/scrapers/debug/list_page.html
Open that file in a browser to see exactly what came back.

FIRST 1500 CHARS OF HTML:

		
		
			
			
			
		
	

<!DOCTYPE html>

<html id="html" lang="en" class="context-client-unknown no-mobile no-js">
<head>
	<meta charset="UTF-8">
	
	
			
		
	<meta name="viewport" content="width=1024">
	<meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1">
	<meta name="description" content="A list of 410 films compiled on Letterboxd, including The &#039;Burbs (1989), 10 Things I Hate About You (1999), 12 Angry Men (1957), 13 Going on 30 (2004) and 50 First 

## §5. List page → film slugs
Walks every `/page/N/` of a list. Tries multiple selectors AND falls back to extracting slugs from `<a href='/film/...'>` anchors if nothing else works.

In [5]:
def extract_slugs_from_soup(soup):
    """Extract film slugs from a list page. Tries every known approach."""
    slugs = []

    # Approach A: data-film-slug attribute (preferred)
    for el in soup.select("[data-film-slug]"):
        s = el.get("data-film-slug")
        if s:
            slugs.append(s)
    if slugs:
        return slugs

    # Approach B: data-target-link like "/film/the-grand-budapest-hotel/"
    for el in soup.select("[data-target-link]"):
        href = el.get("data-target-link", "")
        m = re.match(r"/film/([^/]+)/?", href)
        if m:
            slugs.append(m.group(1))
    if slugs:
        return slugs

    # Approach C: scrape href attributes from poster anchors
    for a in soup.select("a[href^='/film/']"):
        m = re.match(r"/film/([^/]+)/?", a.get("href", ""))
        if m:
            slug = m.group(1)
            # Skip "/film/foo/likes/" style sub-pages
            if "/" not in a.get("href", "")[len("/film/"):].rstrip("/"):
                slugs.append(slug)

    return slugs


def get_film_slugs_from_list(list_url, verbose=True, max_pages=50):
    """Walk every page of a Letterboxd list. Returns ordered list of unique slugs."""
    seen = set()
    ordered = []
    page = 1
    while page <= max_pages:
        url = list_url.rstrip("/") + f"/page/{page}/"
        if verbose:
            print(f"  page {page}: {url}")
        soup = fetch(url)
        if soup is None:
            break

        page_slugs = extract_slugs_from_soup(soup)
        new = 0
        for s in page_slugs:
            if s not in seen:
                seen.add(s)
                ordered.append(s)
                new += 1
        if verbose:
            print(f"    found {len(page_slugs)} on page  ({new} new, {len(ordered)} total)")

        if new == 0:
            # No new films -- we've reached the end
            break

        if not soup.select_one("a.next, .paginate-nextprev a.next"):
            break
        page += 1
    return ordered


## §6. Smoke test — list scraping (single page)
Pulls only page 1 of the first list. **Expect ~100 slugs.**

In [6]:
test_label, test_url = LISTS_TO_SCRAPE[0]
print(f"Testing: {test_label} -> {test_url}\n")
test_soup = fetch(test_url.rstrip("/") + "/page/1/")
if test_soup is None:
    print("FAILED to fetch list page. Run §4 DIAGNOSTIC.")
else:
    slugs = extract_slugs_from_soup(test_soup)
    print(f"Slugs found on page 1: {len(slugs)}")
    print("First 10:")
    for s in slugs[:10]:
        print(" -", s)


Testing: ultimate_comfort -> https://letterboxd.com/cinemaxwell/list/the-ultimate-comfort-movie-list/

Slugs found on page 1: 100
First 10:
 - the-burbs
 - 10-things-i-hate-about-you
 - 12-angry-men
 - 13-going-on-30-3
 - 50-first-dates
 - bud-abbott-and-lou-costello-meet-frankenstein
 - about-time
 - across-the-universe
 - the-adventures-of-priscilla-queen-of-the-desert
 - after-the-thin-man


## §7. Film page → metadata
Letterboxd embeds Movie JSON-LD on every film page — primary source for structured data. HTML fallbacks for runtime, country, language, themes.

In [7]:
def parse_jsonld(soup):
    """Pull the Movie JSON-LD blob from a film page. Returns dict or {}."""
    for tag in soup.find_all("script", {"type": "application/ld+json"}):
        raw = (tag.string or tag.get_text() or "").strip()
        # Letterboxd sometimes wraps JSON-LD in /* CDATA-style */ comments
        raw = raw.lstrip("/*").rstrip("*/").strip()
        try:
            data = json.loads(raw)
        except json.JSONDecodeError:
            continue
        if isinstance(data, dict) and data.get("@type") == "Movie":
            return data
    return {}


def parse_runtime(soup):
    p = soup.select_one("p.text-link.text-footer")
    if not p:
        return None
    m = re.search(r"(\d+)\s*mins?", p.get_text())
    return int(m.group(1)) if m else None


def parse_detail_links(soup, segment):
    """Pull deduped link texts from #tab-details for a path segment."""
    out, seen = [], set()
    for a in soup.select(f"#tab-details a[href*='/films/{segment}/']"):
        text = a.get_text(strip=True)
        if text and text not in seen:
            out.append(text)
            seen.add(text)
    return out


def parse_themes(soup):
    """Themes / mini-themes / nanogenres -- the community tags."""
    out, seen = [], set()
    selectors = [
        "#tab-genres a[href*='/films/theme/']",
        "#tab-genres a[href*='/films/mini-theme/']",
        "#tab-genres a[href*='/films/nanogenre/']",
        "a[href*='/films/theme/']",
        "a[href*='/films/mini-theme/']",
    ]
    for sel in selectors:
        for a in soup.select(sel):
            text = a.get_text(strip=True)
            if text and text not in seen:
                out.append(text)
                seen.add(text)
    return out


def scrape_film(slug):
    """Visit /film/<slug>/ and return a single tidy dict, or None on failure."""
    url = f"{BASE_URL}/film/{slug}/"
    soup = fetch(url)
    if soup is None:
        return None
    jl = parse_jsonld(soup)

    # Director(s)
    directors = []
    d = jl.get("director")
    if isinstance(d, list):
        directors = [x.get("name") for x in d if isinstance(x, dict) and x.get("name")]
    elif isinstance(d, dict) and d.get("name"):
        directors = [d["name"]]

    # Cast (top 10)
    cast = []
    actors = jl.get("actors")
    if isinstance(actors, list):
        cast = [x.get("name") for x in actors if isinstance(x, dict) and x.get("name")][:10]

    # Genres
    genres = jl.get("genre") or []
    if isinstance(genres, str):
        genres = [genres]

    # Year
    year = None
    dp = str(jl.get("datePublished") or "")
    m = re.match(r"(\d{4})", dp)
    if m:
        year = int(m.group(1))
    else:
        h = soup.select_one("small.number a, .releaseyear a")
        if h and h.get_text(strip=True).isdigit():
            year = int(h.get_text(strip=True))

    # Ratings
    avg_rating = num_ratings = num_reviews = None
    agg = jl.get("aggregateRating") or {}
    try:
        if agg.get("ratingValue") is not None:
            avg_rating = float(agg["ratingValue"])
        if agg.get("ratingCount") is not None:
            num_ratings = int(agg["ratingCount"])
        if agg.get("reviewCount") is not None:
            num_reviews = int(agg["reviewCount"])
    except (TypeError, ValueError):
        pass

    # Title (with fallback)
    title = jl.get("name")
    if not title:
        h1 = soup.select_one("h1.headline-1, h1.primaryname, h1")
        title = h1.get_text(strip=True) if h1 else slug

    return {
        "slug": slug,
        "title": title,
        "year": year,
        "runtime_mins": parse_runtime(soup),
        "genres": genres,
        "countries": parse_detail_links(soup, "country"),
        "languages": parse_detail_links(soup, "language"),
        "avg_rating": avg_rating,
        "num_ratings": num_ratings,
        "num_reviews": num_reviews,
        "directors": directors,
        "cast": cast,
        "themes": parse_themes(soup),
        "letterboxd_url": url,
        "date_scraped": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }


## §8. Smoke test — film page scraping

In [8]:
import pprint
test_film = scrape_film("the-grand-budapest-hotel")
pprint.pp(test_film)


{'slug': 'the-grand-budapest-hotel',
 'title': 'Letterboxd — Your life in film',
 'year': None,
 'runtime_mins': 100,
 'genres': [],
 'countries': ['USA', 'Germany'],
 'languages': ['English', 'French'],
 'avg_rating': None,
 'num_ratings': None,
 'num_reviews': None,
 'directors': [],
 'cast': [],
 'themes': ['Crude humor and satire',
            'Relationship comedy',
            'Gags, jokes, and slapstick humor',
            'Amusing jokes and witty satire',
            'Laugh-out-loud relationship entanglements',
            'Funny jokes and crude humor',
            'Dreamlike, quirky, and surreal storytelling'],
 'letterboxd_url': 'https://letterboxd.com/film/the-grand-budapest-hotel/',
 'date_scraped': '2026-05-03T15:31:26+00:00'}


## §9. Full run — Step A: collect slugs across all lists

In [9]:
slug_to_lists = {}
for label, url in LISTS_TO_SCRAPE:
    print(f"\n=== Walking list: {label} ===")
    slugs = get_film_slugs_from_list(url)
    print(f"   -> {len(slugs)} films on '{label}'")
    for s in slugs:
        slug_to_lists.setdefault(s, set()).add(label)
print(f"\n=== {len(slug_to_lists)} unique films across all lists ===")



=== Walking list: ultimate_comfort ===
  page 1: https://letterboxd.com/cinemaxwell/list/the-ultimate-comfort-movie-list/page/1/
    found 100 on page  (100 new, 100 total)
  page 2: https://letterboxd.com/cinemaxwell/list/the-ultimate-comfort-movie-list/page/2/
    found 100 on page  (100 new, 200 total)
  page 3: https://letterboxd.com/cinemaxwell/list/the-ultimate-comfort-movie-list/page/3/
    found 100 on page  (100 new, 300 total)
  page 4: https://letterboxd.com/cinemaxwell/list/the-ultimate-comfort-movie-list/page/4/
    found 100 on page  (100 new, 400 total)
  page 5: https://letterboxd.com/cinemaxwell/list/the-ultimate-comfort-movie-list/page/5/
    found 10 on page  (10 new, 410 total)
   -> 410 films on 'ultimate_comfort'

=== Walking list: comfort_izzy ===
  page 1: https://letterboxd.com/cinemaparadisa/list/comfort-movies/page/1/
    found 100 on page  (100 new, 100 total)
  page 2: https://letterboxd.com/cinemaparadisa/list/comfort-movies/page/2/
    found 83 on page  

## §9. Full run — Step B: scrape each film page
Slow part. Checkpoints to CSV every 25 films.

In [10]:
LIST_COLUMNS = ["genres", "countries", "languages", "directors", "cast", "themes", "lists"]


def write_csv(rows, path):
    df_local = pd.DataFrame(rows)
    for col in LIST_COLUMNS:
        if col in df_local.columns:
            df_local[col] = df_local[col].apply(lambda xs: "|".join(xs) if isinstance(xs, list) else "")
    df_local.to_csv(path, index=False, quoting=csv.QUOTE_MINIMAL)
    return df_local


PROGRESS_EVERY = 25
rows = []
total = len(slug_to_lists)
for i, (slug, list_labels) in enumerate(slug_to_lists.items(), start=1):
    print(f"[{i}/{total}] {slug}")
    film = scrape_film(slug)
    if not film:
        continue
    film["lists"] = sorted(list_labels)
    film["num_lists_appeared_in"] = len(list_labels)
    rows.append(film)
    if i % PROGRESS_EVERY == 0:
        write_csv(rows, OUTPUT_CSV)
        print(f"   [checkpoint] {len(rows)} rows -> {OUTPUT_CSV}")
df = write_csv(rows, OUTPUT_CSV)
print(f"\nDone. Wrote {len(df)} rows to {OUTPUT_CSV}")


[1/674] the-burbs
[2/674] 10-things-i-hate-about-you
[3/674] 12-angry-men
[4/674] 13-going-on-30-3
[5/674] 50-first-dates
[6/674] bud-abbott-and-lou-costello-meet-frankenstein
[7/674] about-time
[8/674] across-the-universe
[9/674] the-adventures-of-priscilla-queen-of-the-desert
[10/674] after-the-thin-man
[11/674] airplane
[12/674] akira
[13/674] aladdin
[14/674] alien
[15/674] aliens
[16/674] almost-famous
[17/674] american-hustle
[18/674] an-american-tail
[19/674] anastasia-1997
[20/674] anna-and-the-apocalypse
[21/674] annie-hall
[22/674] the-apartment
[23/674] apollo-13
[24/674] aquaman-2018
[25/674] arrival-2016
   [checkpoint] 25 rows -> data/comfort_feelgood_films.csv
[26/674] avengers-endgame
[27/674] avengers-infinity-war
[28/674] back-to-the-future
[29/674] back-to-the-future-part-ii
[30/674] barefoot-in-the-park
[31/674] batman-begins
[32/674] batman-mask-of-the-phantasm
[33/674] beauty-and-the-beast-1991
[34/674] bedknobs-and-broomsticks
[35/674] before-sunset
[36/674] best

## §10. Inspect the dataset

In [11]:
df = pd.read_csv(OUTPUT_CSV)
print(f"Shape: {df.shape}")
print(f"\nMissing values per column:")
print(df.isna().sum())
df.head(10)


Shape: (674, 17)

Missing values per column:
slug                       0
title                      0
year                     674
runtime_mins               0
genres                   674
countries                  1
languages                  0
avg_rating               674
num_ratings              674
num_reviews              674
directors                674
cast                     674
themes                     9
letterboxd_url             0
date_scraped               0
lists                      0
num_lists_appeared_in      0
dtype: int64


,slug,title,year,runtime_mins,genres,countries,languages,avg_rating,num_ratings,num_reviews,directors,cast,themes,letterboxd_url,date_scraped,lists,num_lists_appeared_in
0,the-burbs,Letterboxd — Your life in film,NaN,102,NaN,USA,English,NaN,NaN,NaN,NaN,NaN,"Crude humor and satire|Gags, jokes, and slapst...",https://letterboxd.com/film/the-burbs/,2026-05-03T15:31:51+00:00,ultimate_comfort,1
1,10-things-i-hate-about-you,Letterboxd — Your life in film,NaN,97,NaN,USA,English|French,NaN,NaN,NaN,NaN,NaN,Underdogs and coming of age|Relationship comed...,https://letterboxd.com/film/10-things-i-hate-a...,2026-05-03T15:31:53+00:00,comfort_izzy|ultimate_comfort,2
2,12-angry-men,Letterboxd — Your life in film,NaN,97,NaN,USA,English,NaN,NaN,NaN,NaN,NaN,"Humanity and the world around us|Gripping, int...",https://letterboxd.com/film/12-angry-men/,2026-05-03T15:31:55+00:00,ultimate_comfort,1
3,13-going-on-30-3,Letterboxd — Your life in film,NaN,98,NaN,USA,English,NaN,NaN,NaN,NaN,NaN,Relationship comedy|Underdogs and coming of ag...,https://letterboxd.com/film/13-going-on-30-3/,2026-05-03T15:31:57+00:00,comfort_izzy|heartwarming|ultimate_comfort,3
4,50-first-dates,Letterboxd — Your life in film,NaN,99,NaN,USA,English,NaN,NaN,NaN,NaN,NaN,Relationship comedy|Crude humor and satire|Mov...,https://letterboxd.com/film/50-first-dates/,2026-05-03T15:31:59+00:00,heartwarming|ultimate_comfort,2
5,bud-abbott-and-lou-costello-meet-frankenstein,Letterboxd — Your life in film,NaN,83,NaN,USA,English,NaN,NaN,NaN,NaN,NaN,"Crude humor and satire|Horror, the undead and ...",https://letterboxd.com/film/bud-abbott-and-lou...,2026-05-03T15:32:02+00:00,ultimate_comfort,1
6,about-time,Letterboxd — Your life in film,NaN,123,NaN,UK|USA,English,NaN,NaN,NaN,NaN,NaN,Relationship comedy|Moving relationship storie...,https://letterboxd.com/film/about-time/,2026-05-03T15:32:04+00:00,comfort_izzy|heartwarming|ultimate_comfort,3
7,across-the-universe,Letterboxd — Your life in film,NaN,133,NaN,USA,English,NaN,NaN,NaN,NaN,NaN,Song and dance|Humanity and the world around u...,https://letterboxd.com/film/across-the-universe/,2026-05-03T15:32:06+00:00,ultimate_comfort,1
8,the-adventures-of-priscilla-queen-of-the-desert,Letterboxd — Your life in film,NaN,103,NaN,Australia|USA,English,NaN,NaN,NaN,NaN,NaN,"Crude humor and satire|Song and dance|Gags, jo...",https://letterboxd.com/film/the-adventures-of-...,2026-05-03T15:32:08+00:00,ultimate_comfort,1
9,after-the-thin-man,Letterboxd — Your life in film,NaN,112,NaN,USA,English,NaN,NaN,NaN,NaN,NaN,Thrillers and murder mysteries|Crude humor and...,https://letterboxd.com/film/after-the-thin-man/,2026-05-03T15:32:10+00:00,ultimate_comfort,1


In [12]:
print("Avg rating distribution:")
print(df["avg_rating"].describe())
print("\nTop 20 themes across the corpus:")
top_themes = df["themes"].dropna().str.split("|").explode().str.strip().value_counts().head(20)
print(top_themes)


Avg rating distribution:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: avg_rating, dtype: float64

Top 20 themes across the corpus:
themes
Crude humor and satire                                237
Moving relationship stories                           203
Laugh-out-loud relationship entanglements             172
Heartfelt and sentimental family stories              171
Charming romances and delightful chemistry            156
Gags, jokes, and slapstick humor                      151
Relationship comedy                                   150
Funny jokes and crude humor                           121
Humanity and the world around us                      115
Amusing jokes and witty satire                        100
Kids' animated fun and adventure                      100
Quirky and endearing relationships                    100
Song and dance                                         98
Emotional and heartfelt family dramas      